# ROC Analysis for L2FC Threshold Selection

This notebook performs ROC analysis to identify an optimal L2FC threshold that achieves FDR < 0.1.

**Input files required:**
- **True positives**: membrane proteins (column 1: protein name, column 2: L2FC)
- **False positives**: nuclear proteins (column 1: protein name, column 2: L2FC)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.metrics import auc

## File Paths
Edit the paths below to point to your Excel files.

In [ ]:
# --- USER INPUT: set paths to your Excel files ---
TRUE_POSITIVE_PATH  = "~/pfcnrxn_TPlist_proteome.xlsx"   # true positives (membrane proteins)
FALSE_POSITIVE_PATH = "~/pfcnrxn_FPlist_proteome.xlsx"  # false positives (nuclear proteins)

# Experiment name: auto-derived from the TP filename stem; override manually if needed
# e.g., EXPERIMENT_NAME = "pfcnrxn"
EXPERIMENT_NAME = Path(TRUE_POSITIVE_PATH).stem

# Output folder: created automatically next to this notebook
OUT_DIR = Path(os.path.dirname(os.path.abspath(TRUE_POSITIVE_PATH))) / EXPERIMENT_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment : {EXPERIMENT_NAME}")
print(f"Output dir : {OUT_DIR}")

## Load Data

In [ ]:
tp_df = pd.read_excel(TRUE_POSITIVE_PATH,  header=0, usecols=[0, 1])
fp_df = pd.read_excel(FALSE_POSITIVE_PATH, header=0, usecols=[0, 1])

tp_df.columns = ["protein", "L2FC"]
fp_df.columns = ["protein", "L2FC"]

tp_df = tp_df.dropna(subset=["L2FC"])
fp_df = fp_df.dropna(subset=["L2FC"])

n_tp_total = len(tp_df)   # total membrane proteins (P = condition positive)
n_fp_total = len(fp_df)   # total nuclear proteins  (N = condition negative)

print(f"True positives (membrane proteins): {n_tp_total}")
print(f"False positives (nuclear proteins): {n_fp_total}")
print(f"\nTrue positive L2FC range:  [{tp_df.L2FC.min():.3f}, {tp_df.L2FC.max():.3f}]")
print(f"False positive L2FC range: [{fp_df.L2FC.min():.3f}, {fp_df.L2FC.max():.3f}]")

## ROC Analysis

At each L2FC threshold, proteins with L2FC ≥ threshold are called "positive" (predicted membrane protein).

| Metric | Formula |
|--------|--------|
| TPR (sensitivity) | TP / (TP + FN) |
| FPR (1 − specificity) | FP / (FP + TN) |
| FDR | FP / (FP + TP) |

In [ ]:
all_l2fc = np.sort(np.unique(np.concatenate([tp_df.L2FC.values, fp_df.L2FC.values])))
# Sweep thresholds from just below min to just above max
thresholds = np.linspace(all_l2fc.min() - 0.01, all_l2fc.max() + 0.01, 500)

results = []
for t in thresholds:
    TP = (tp_df.L2FC >= t).sum()   # membrane proteins called positive
    FN = (tp_df.L2FC <  t).sum()   # membrane proteins missed
    FP = (fp_df.L2FC >= t).sum()   # nuclear proteins called positive
    TN = (fp_df.L2FC <  t).sum()   # nuclear proteins correctly excluded

    TPR = TP / n_tp_total if n_tp_total > 0 else 0
    FPR = FP / n_fp_total if n_fp_total > 0 else 0
    FDR = FP / (FP + TP) if (FP + TP) > 0 else 0

    results.append({"threshold": t, "TP": TP, "FP": FP, "TN": TN, "FN": FN,
                    "TPR": TPR, "FPR": FPR, "FDR": FDR})

res = pd.DataFrame(results)
roc_auc = auc(res.FPR, res.TPR)
print(f"AUROC = {roc_auc:.4f}")

## Identify Optimal Threshold (FDR < 0.1)

In [ ]:
FDR_TARGET = 0.1

# Among rows that satisfy FDR < target, pick the one with the lowest (most permissive) threshold
# This maximises sensitivity while keeping FDR controlled
fdr_ok = res[res.FDR < FDR_TARGET]

if fdr_ok.empty:
    print(f"No threshold achieves FDR < {FDR_TARGET}. Consider relaxing the target.")
    optimal = None
else:
    optimal = fdr_ok.loc[fdr_ok.threshold.idxmin()]   # lowest threshold that still meets FDR
    print(f"Optimal L2FC threshold : {optimal.threshold:.4f}")
    print(f"  TPR (sensitivity)    : {optimal.TPR:.4f}  ({int(optimal.TP)}/{n_tp_total} membrane proteins recovered)")
    print(f"  FPR (1-specificity)  : {optimal.FPR:.4f}  ({int(optimal.FP)}/{n_fp_total} nuclear proteins leaked)")
    print(f"  FDR                  : {optimal.FDR:.4f}")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Panel 1: ROC curve ---
ax = axes[0]
ax.plot(res.FPR, res.TPR, color="steelblue", lw=2, label=f"ROC (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Random classifier")
if optimal is not None:
    ax.scatter(optimal.FPR, optimal.TPR, color="crimson", zorder=5, s=80,
               label=f"Threshold = {optimal.threshold:.3f}\n(FDR = {optimal.FDR:.3f})")
ax.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax.set_title("ROC Curve", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# --- Panel 2: FDR vs L2FC threshold ---
ax2 = axes[1]
ax2.plot(res.threshold, res.FDR, color="darkorange", lw=2, label="FDR")
ax2.axhline(FDR_TARGET, color="crimson", linestyle="--", lw=1.5, label=f"FDR = {FDR_TARGET}")
if optimal is not None:
    ax2.axvline(optimal.threshold, color="steelblue", linestyle=":", lw=1.5,
                label=f"L2FC = {optimal.threshold:.3f}")
ax2.set_xlabel("L2FC Threshold", fontsize=12)
ax2.set_ylabel("FDR", fontsize=12)
ax2.set_title("FDR vs L2FC Threshold", fontsize=13)
ax2.set_ylim(-0.02, 1.02)
ax2.legend(fontsize=10)

plt.tight_layout()
out_path = OUT_DIR / f"{EXPERIMENT_NAME}_ROC_FDR.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## Bidirectional L2FC Distribution

Bars above zero = true positive membrane proteins; bars below zero = false positive nuclear proteins.
The optimal L2FC threshold (dashed line) and original 0.585 cutoff (dotted line) are marked.

In [ ]:
bin_width = 0.25
l2fc_min = min(tp_df.L2FC.min(), fp_df.L2FC.min())
l2fc_max = max(tp_df.L2FC.max(), fp_df.L2FC.max())
bins = np.arange(np.floor(l2fc_min / bin_width) * bin_width,
                 np.ceil(l2fc_max  / bin_width) * bin_width + bin_width,
                 bin_width)

tp_counts, _ = np.histogram(tp_df.L2FC, bins=bins)
fp_counts, _ = np.histogram(fp_df.L2FC, bins=bins)
bin_centers  = (bins[:-1] + bins[1:]) / 2

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(bin_centers,  tp_counts, width=bin_width * 0.9,
       color="steelblue", alpha=0.85, label="True positives (membrane proteins)")
ax.bar(bin_centers, -fp_counts, width=bin_width * 0.9,
       color="tomato",    alpha=0.85, label="False positives (nuclear proteins)")

if optimal is not None:
    ax.axvline(optimal.threshold, color="black", linestyle="--", lw=1.8,
               label=f"Optimal threshold = {optimal.threshold:.3f} (FDR = {optimal.FDR:.3f})")

ax.axvline(0.585, color="grey", linestyle=":", lw=1.5, label="Original cutoff = 0.585")

y_max = max(tp_counts.max(), fp_counts.max()) * 1.15
ax.set_ylim(-y_max, y_max)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: str(int(abs(y)))))
ax.axhline(0, color="black", lw=0.8)

ax.set_xlabel("Log₂ Fold Change (L2FC)", fontsize=12)
ax.set_ylabel("Protein count", fontsize=12)
ax.set_title("Bidirectional L2FC Distribution", fontsize=13)
ax.legend(fontsize=10)

ax.text(0.01, 0.97, "▲ Membrane proteins (TP)",
        transform=ax.transAxes, va="top", color="steelblue", fontsize=9)
ax.text(0.01, 0.03, "▼ Nuclear proteins (FP)",
        transform=ax.transAxes, va="bottom", color="tomato", fontsize=9)

plt.tight_layout()
out_path = OUT_DIR / f"{EXPERIMENT_NAME}_bidirectional_histogram.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

## Summary Table

In [ ]:
# Show key statistics at a handful of informative thresholds
landmarks = [0.0, 0.585, 1.0, 1.5, 2.0]
rows = []
for t in landmarks:
    TP = (tp_df.L2FC >= t).sum()
    FP = (fp_df.L2FC >= t).sum()
    FN = (tp_df.L2FC <  t).sum()
    TN = (fp_df.L2FC <  t).sum()
    TPR = TP / n_tp_total if n_tp_total > 0 else 0
    FPR = FP / n_fp_total if n_fp_total > 0 else 0
    FDR = FP / (FP + TP) if (FP + TP) > 0 else 0
    rows.append({"L2FC threshold": t, "TP": TP, "FP": FP, "FN": FN, "TN": TN,
                 "TPR": round(TPR, 4), "FPR": round(FPR, 4), "FDR": round(FDR, 4)})

summary = pd.DataFrame(rows)
if optimal is not None:
    opt_row = {"L2FC threshold": round(optimal.threshold, 4),
               "TP": int(optimal.TP), "FP": int(optimal.FP),
               "FN": int(optimal.FN), "TN": int(optimal.TN),
               "TPR": round(optimal.TPR, 4), "FPR": round(optimal.FPR, 4),
               "FDR": round(optimal.FDR, 4)}
    summary = pd.concat([summary, pd.DataFrame([opt_row])], ignore_index=True)
    summary = summary.sort_values("L2FC threshold").reset_index(drop=True)

csv_path = OUT_DIR / f"{EXPERIMENT_NAME}_summary_table.csv"
summary.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
display(summary)

In [ ]:
# --- FDR at a user-specified L2FC cutoff ---
# Enter the L2FC threshold you want to evaluate:
L2FC_CUTOFF = 1.554

TP = int((tp_df.L2FC >= L2FC_CUTOFF).sum())   # membrane proteins called positive
FP = int((fp_df.L2FC >= L2FC_CUTOFF).sum())   # nuclear proteins called positive
FN = int((tp_df.L2FC <  L2FC_CUTOFF).sum())   # membrane proteins missed
TN = int((fp_df.L2FC <  L2FC_CUTOFF).sum())   # nuclear proteins correctly excluded

TPR = TP / n_tp_total if n_tp_total > 0 else 0
FPR = FP / n_fp_total if n_fp_total > 0 else 0
FDR = FP / (FP + TP) if (FP + TP) > 0 else float("nan")

print(f"L2FC cutoff = {L2FC_CUTOFF}")
print(f"  TP = {TP}   FP = {FP}   FN = {FN}   TN = {TN}")
print(f"  TPR (sensitivity)   = {TPR:.4f}  ({TP}/{n_tp_total} membrane proteins recovered)")
print(f"  FPR (1-specificity) = {FPR:.4f}  ({FP}/{n_fp_total} nuclear proteins leaked)")
print(f"  FDR = {FDR:.4f}")